In [0]:
# ============================================================================
# Avaliação de Qualidade de Dados — silver.aerodromos
# ============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

TABELA = "silver.aerodromos"
df = spark.table(TABELA)

total_linhas = df.count()
print(f"\n{'='*70}")
print(f"  AVALIAÇÃO DE QUALIDADE DE DADOS — {TABELA}")
print(f"  Total de registros: {total_linhas:,}")
print(f"{'='*70}\n")

# ---------------------------------------------------------------------------
# 1. Esquema da tabela
# ---------------------------------------------------------------------------
print("1. ESQUEMA DA TABELA")
print("-" * 70)
df.printSchema()

# ---------------------------------------------------------------------------
# 2. Amostra de dados
# ---------------------------------------------------------------------------
print("\n2. AMOSTRA DE DADOS (10 linhas)")
print("-" * 70)
display(df.limit(10))

# ---------------------------------------------------------------------------
# 3. Valores nulos por coluna
# ---------------------------------------------------------------------------
print("\n3. CONTAGEM DE VALORES NULOS POR COLUNA")
print("-" * 70)

colunas = df.columns
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in colunas
]).collect()[0]

print(f"{'Coluna':<40} {'Nulos':>10} {'% Nulos':>10}")
print(f"{'-'*40} {'-'*10} {'-'*10}")
for c in colunas:
    n = null_counts[c]
    pct = (n / total_linhas * 100) if total_linhas > 0 else 0
    flag = " <-- ATENÇÃO" if pct > 0 else ""
    print(f"{c:<40} {n:>10,} {pct:>9.2f}%{flag}")

# ---------------------------------------------------------------------------
# 4. Duplicidade de registros (linha inteira)
# ---------------------------------------------------------------------------
print("\n4. DUPLICIDADE DE REGISTROS (linha inteira)")
print("-" * 70)
total_distintos = df.distinct().count()
duplicados = total_linhas - total_distintos
print(f"  Registros distintos : {total_distintos:,}");
print(f"  Registros duplicados: {duplicados:,}")
if duplicados > 0:
    print("  <-- ATENÇÃO: existem registros completamente duplicados")
    display(df.groupBy(colunas).count().filter("count > 1").orderBy(F.desc("count")).limit(20))

# ---------------------------------------------------------------------------
# 5. Duplicidade em chaves prováveis (codigo_oaci / icao_code / codigo)
# ---------------------------------------------------------------------------
print("\n5. DUPLICIDADE EM CHAVES PROVÁVEIS")
print("-" * 70)
chaves_candidatas = [c for c in colunas if c.lower() in (
    "codigo_oaci", "icao_code", "icao", "codigo", "codigo_icao",
    "oaci", "codigo_iata", "iata_code", "iata"
)]

if chaves_candidatas:
    for chave in chaves_candidatas:
        n_dups = df.groupBy(chave).count().filter("count > 1").count()
        n_nulls = df.filter(F.col(chave).isNull()).count()
        print(f"  Coluna '{chave}': {n_dups} valor(es) duplicado(s), {n_nulls} nulo(s)")
        if n_dups > 0:
            display(df.groupBy(chave).count().filter("count > 1").orderBy(F.desc("count")).limit(20))
else:
    print("  Nenhuma coluna candidata a chave encontrada automaticamente.")
    print(f"  Colunas disponíveis: {colunas}")

# ---------------------------------------------------------------------------
# 6. Estatísticas descritivas (colunas numéricas)
# ---------------------------------------------------------------------------
print("\n6. ESTATÍSTICAS DESCRITIVAS (colunas numéricas)")
print("-" * 70)
colunas_numericas = [f.name for f in df.schema.fields if isinstance(f.dataType, (IntegerType, LongType, FloatType, DoubleType, DecimalType))]
if colunas_numericas:
    display(df.select(colunas_numericas).describe())
else:
    print("  Nenhuma coluna numérica encontrada.")

# ---------------------------------------------------------------------------
# 7. Cardinalidade e valores distintos (colunas categóricas / texto)
# ---------------------------------------------------------------------------
print("\n7. CARDINALIDADE DE COLUNAS DE TEXTO/CATEGÓRICAS")
print("-" * 70)
colunas_texto = [f.name for f in df.schema.fields if isinstance(f.dataType, (StringType, BooleanType))]
print(f"{'Coluna':<40} {'Distintos':>12}")
print(f"{'-'*40} {'-'*12}")
for c in colunas_texto:
    n = df.select(c).distinct().count()
    print(f"{c:<40} {n:>12,}")
    # Mostrar valores mais frequentes se cardinalidade for baixa
    if 0 < n <= 30:
        top = df.groupBy(c).count().orderBy(F.desc("count")).limit(30).collect()
        for row in top:
            print(f"    -> {row[c]!s:<35} {row['count']:>10,}")

# ---------------------------------------------------------------------------
# 8. Detecção de strings vazias ou com apenas espaços
# ---------------------------------------------------------------------------
print("\n8. STRINGS VAZIAS OU COM APENAS ESPAÇOS")
print("-" * 70)
for c in colunas_texto:
    vazias = df.filter((F.col(c) == "") | (F.trim(F.col(c)) == "")).count()
    if vazias > 0:
        print(f"  Coluna '{c}': {vazias} registro(s) com string vazia/espaços")

# ---------------------------------------------------------------------------
# 9. Valores extremos em coordenadas (se existirem)
# ---------------------------------------------------------------------------
print("\n9. VALIDAÇÃO DE COORDENADAS GEOGRÁFICAS")
print("-" * 70)
colunas_coord = [c for c in colunas if any(k in c.lower() for k in ("latitude", "longitude", "lat", "lon", "latitud", "longitud"))]
for c in colunas_coord:
    if "lat" in c.lower():
        fora = df.filter((F.col(c) < -90) | (F.col(c) > 90)).count()
        print(f"  {c}: {fora} valor(es) fora do intervalo [-90, 90]")
    elif "lon" in c.lower():
        fora = df.filter((F.col(c) < -180) | (F.col(c) > 180)).count()
        print(f"  {c}: {fora} valor(es) fora do intervalo [-180, 180]")
    display(df.select(c).describe())

# ---------------------------------------------------------------------------
# 10. Resumo e recomendações de tratamento
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("  RESUMO E RECOMENDAÇÕES DE TRATAMENTO DE QUALIDADE")
print("="*70)

problemas = []

# Nulls
for c in colunas:
    n = null_counts[c]
    if n > 0:
        problemas.append(("Nulos", c, n))

# Duplicidade
if duplicados > 0:
    problemas.append(("Duplicidade (linha inteira)", "—", duplicados))

for chave in chaves_candidatas:
    n_dups = df.groupBy(chave).count().filter("count > 1").count()
    if n_dups > 0:
        problemas.append(("Duplicidade de chave", chave, n_dups))

if problemas:
    for tipo, col, n in problemas:
        print(f"\n  [{tipo}] Coluna: {col} | Registros afetados: {n:,}")
else:
    print("\n  Nenhum problema de qualidade óbvio detectado.")

print("\n  RECOMENDAÇÕES DE TRATAMENTO:")
print("  1. Nulos: padronizar como NULL (não vazio), documentar regra de preenchimento.")
print("  2. Duplicados: usar dropDuplicates() ou window com row_number para deduplicar.")
print("  3. Chaves: garantir unicidade do código OACI/IATA; criar constraint ou check.")
print("  4. Strings: aplicar trim() e normalizar case (upper) em colunas categóricas.")
print("  5. Coordenadas: validar intervalo válido; remover/invalidar coordenadas fora do range.")
print("  6. Adicionar tags e comentários no Unity Catalog para governança.")
print("  7. Implementar verificações com EXPECTATIONS em pipelines (DQ rules).")
print("="*70)